In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Mount Google Drive

The previous error `FileNotFoundError` suggests that the Google Drive containing your dataset (`/content/drive/MyDrive/umurinzi_data/CICIDS2017/`) might not be mounted. Please run the cell above to mount your Google Drive, then re-run the data loading cells.

In [2]:
import os
import gc
import numpy as np
import pandas as pd

RANDOM_STATE = 42
ROWS_PER_DATASET = 100_000

CICIDS_FOLDER = '/content/drive/MyDrive/umurinzi_data/CICIDS2017/'
UNSW_FOLDER = '/content/drive/MyDrive/umurinzi_data/UNSW-NB15/'
OUTPUT_FOLDER = '/content/drive/MyDrive/umurinzi_data/output/'

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [3]:
def get_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def stratified_sample(df, label_col, n_rows, random_state=42):
    if len(df) <= n_rows:
        return df.reset_index(drop=True)

    sampled = (
        df.groupby(label_col, group_keys=False)
          .apply(lambda x: x.sample(
              n=max(1, int(len(x) / len(df) * n_rows)),
              random_state=random_state
          ))
          .reset_index(drop=True)
    )

    if len(sampled) > n_rows:
        sampled = sampled.sample(n=n_rows, random_state=random_state)

    return sampled.reset_index(drop=True)

In [4]:
cicids_files = [
    'Monday-WorkingHours.pcap_ISCX.csv',
    'Tuesday-WorkingHours.pcap_ISCX.csv',
    'Wednesday-workingHours.pcap_ISCX.csv',                      # lowercase 'w'
    'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
    'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv', # 'Infilteration' spelling
    'Friday-WorkingHours-Morning.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
]

cicids_frames = []

for filename in cicids_files:
    filepath = os.path.join(CICIDS_FOLDER, filename)
    print(f'Loading CICIDS file: {filename}')

    df = pd.read_csv(filepath, low_memory=False)
    df.columns = df.columns.str.strip()

    label_col = [c for c in df.columns if 'label' in c.lower()][0]
    df = df.rename(columns={label_col: 'original_label'})

    cicids_frames.append(df)

cicids_raw = pd.concat(cicids_frames, ignore_index=True)

del cicids_frames
gc.collect()

print(f'CICIDS loaded: {len(cicids_raw):,} rows')
print(cicids_raw['original_label'].value_counts())

Loading CICIDS file: Monday-WorkingHours.pcap_ISCX.csv
Loading CICIDS file: Tuesday-WorkingHours.pcap_ISCX.csv
Loading CICIDS file: Wednesday-workingHours.pcap_ISCX.csv
Loading CICIDS file: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading CICIDS file: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading CICIDS file: Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading CICIDS file: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Loading CICIDS file: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
CICIDS loaded: 2,830,743 rows
original_label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web 

In [5]:
print('Cleaning CICIDS2017...')

df_c = cicids_raw.copy()
df_c.columns = df_c.columns.str.strip()

df_c['original_label'] = df_c['original_label'].astype(str).str.strip()

df_c = df_c.replace([np.inf, -np.inf], np.nan)
df_c = df_c.dropna()
df_c = df_c.drop_duplicates()

print(f'CICIDS cleaned: {len(df_c):,} rows')
print(df_c['original_label'].value_counts())

del cicids_raw
gc.collect()

Cleaning CICIDS2017...
CICIDS cleaned: 2,520,798 rows
original_label
BENIGN                        2095057
DoS Hulk                       172846
DDoS                           128014
PortScan                        90694
DoS GoldenEye                   10286
FTP-Patator                      5931
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1948
Web Attack � Brute Force         1470
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


11

In [6]:
cicids_label_map = {
    'BENIGN': 'NORMAL',

    'DoS Hulk': 'DoS',
    'DoS slowloris': 'DoS',
    'DoS Slowhttptest': 'DoS',
    'DoS GoldenEye': 'DoS',
    'Heartbleed': 'DoS',
    'DDoS': 'DoS',
    'DDos': 'DoS',

    'PortScan': 'Probe',

    'FTP-Patator': 'Exploitation',
    'SSH-Patator': 'Exploitation',

    'Web Attack \\x96 Brute Force': 'Exploitation',
    'Web Attack \\x96 XSS': 'Exploitation',
    'Web Attack \\x96 Sql Injection': 'Exploitation',

    'Web Attack – Brute Force': 'Exploitation',
    'Web Attack – XSS': 'Exploitation',
    'Web Attack – Sql Injection': 'Exploitation',

    'Web Attack - Brute Force': 'Exploitation',
    'Web Attack - XSS': 'Exploitation',
    'Web Attack - Sql Injection': 'Exploitation',

    'Web Attack � Brute Force': 'Exploitation',
    'Web Attack � XSS': 'Exploitation',
    'Web Attack � Sql Injection': 'Exploitation',

    'Bot': 'Backdoor',
    'Infiltration': 'Backdoor',
}

df_c['unified_label'] = df_c['original_label'].map(cicids_label_map)

if df_c['unified_label'].isna().sum() > 0:
    print('Unmapped CICIDS labels:')
    print(df_c[df_c['unified_label'].isna()]['original_label'].value_counts())

df_c = df_c.dropna(subset=['unified_label']).reset_index(drop=True)

print(df_c['unified_label'].value_counts())

unified_label
NORMAL          2095057
DoS              321770
Probe             90694
Exploitation      11293
Backdoor           1984
Name: count, dtype: int64


In [7]:
df_c = stratified_sample(
    df_c,
    label_col='unified_label',
    n_rows=ROWS_PER_DATASET,
    random_state=RANDOM_STATE
)

print(f'CICIDS sampled: {len(df_c):,} rows')
print(df_c['unified_label'].value_counts())

CICIDS sampled: 99,996 rows
unified_label
NORMAL          83110
DoS             12764
Probe            3597
Exploitation      447
Backdoor           78
Name: count, dtype: int64


/tmp/ipykernel_1406/3033405128.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


In [8]:
print('Extracting CICIDS canonical features...')

cicids_cols = {
    'dest_port'         : get_col(df_c, ['Destination Port', 'Dst Port']),
    'protocol'          : get_col(df_c, ['Protocol']),
    'flow_duration'     : get_col(df_c, ['Flow Duration']),

    'fwd_pkts'          : get_col(df_c, ['Total Fwd Packets', 'Tot Fwd Pkts']),
    'bwd_pkts'          : get_col(df_c, ['Total Backward Packets', 'Tot Bwd Pkts']),

    'fwd_bytes'         : get_col(df_c, ['Total Length of Fwd Packets', 'TotLen Fwd Pkts']),
    'bwd_bytes'         : get_col(df_c, ['Total Length of Bwd Packets', 'TotLen Bwd Pkts']),

    'fwd_pkt_len_mean'  : get_col(df_c, ['Fwd Packet Length Mean', 'Fwd Pkt Len Mean']),
    'bwd_pkt_len_mean'  : get_col(df_c, ['Bwd Packet Length Mean', 'Bwd Pkt Len Mean']),

    'fwd_iat_mean'      : get_col(df_c, ['Fwd IAT Mean']),
    'bwd_iat_mean'      : get_col(df_c, ['Bwd IAT Mean']),
    'fwd_iat_std'       : get_col(df_c, ['Fwd IAT Std']),
    'bwd_iat_std'       : get_col(df_c, ['Bwd IAT Std']),

    'flow_bytes_per_sec': get_col(df_c, ['Flow Bytes/s', 'Flow Bytes_s']),
    'flow_pkts_per_sec' : get_col(df_c, ['Flow Packets/s', 'Flow Packets_s']),

    'syn_flag_count'    : get_col(df_c, ['SYN Flag Count', 'SYN Flag Cnt']),
    'ack_flag_count'    : get_col(df_c, ['ACK Flag Count', 'ACK Flag Cnt']),
    'rst_flag_count'    : get_col(df_c, ['RST Flag Count', 'RST Flag Cnt']),
    'psh_flag_count'    : get_col(df_c, ['PSH Flag Count', 'PSH Flag Cnt']),

    'fwd_win'           : get_col(df_c, ['Init_Win_bytes_forward', 'Init Fwd Win Byts']),
    'bwd_win'           : get_col(df_c, ['Init_Win_bytes_backward', 'Init Bwd Win Byts']),
}

cicids_features = pd.DataFrame(index=df_c.index)

for canonical_name, original_col in cicids_cols.items():
    if original_col is not None:
        cicids_features[canonical_name] = df_c[original_col]
    else:
        cicids_features[canonical_name] = 0

cicids_features['fwd_load'] = cicids_features['flow_bytes_per_sec']
cicids_features['bwd_load'] = cicids_features['flow_bytes_per_sec']

cicids_features['ct_srv_src'] = 0
cicids_features['ct_dst_ltm'] = 0

cicids_features['unified_label'] = df_c['unified_label'].values
cicids_features['dataset_source'] = 0

for col in cicids_features.columns:
    if col not in ['unified_label']:
        cicids_features[col] = pd.to_numeric(cicids_features[col], errors='coerce')

cicids_features = cicids_features.replace([np.inf, -np.inf], np.nan)
cicids_features = cicids_features.dropna().reset_index(drop=True)

print(cicids_features.shape)
print(cicids_features['unified_label'].value_counts())

del df_c
gc.collect()

Extracting CICIDS canonical features...
(99996, 27)
unified_label
NORMAL          83110
DoS             12764
Probe            3597
Exploitation      447
Backdoor           78
Name: count, dtype: int64


0

In [9]:
unsw_column_names = [
    'srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur',
    'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service',
    'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb',
    'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len',
    'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 'Dintpkt',
    'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl',
    'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src',
    'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm',
    'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'label'
]

unsw_files = [
    'UNSW-NB15_1.csv',
    'UNSW-NB15_2.csv',
    'UNSW-NB15_3.csv',
    'UNSW-NB15_4.csv',
]

unsw_frames = []

for filename in unsw_files:
    filepath = os.path.join(UNSW_FOLDER, filename)
    print(f'Loading UNSW file: {filename}')

    df = pd.read_csv(
        filepath,
        header=None,
        names=unsw_column_names,
        low_memory=False
    )

    if str(df.iloc[0]['srcip']).lower() == 'srcip':
        df = df.iloc[1:].reset_index(drop=True)

    unsw_frames.append(df)

unsw_raw = pd.concat(unsw_frames, ignore_index=True)

del unsw_frames
gc.collect()

print(f'UNSW loaded: {len(unsw_raw):,} rows')

Loading UNSW file: UNSW-NB15_1.csv
Loading UNSW file: UNSW-NB15_2.csv
Loading UNSW file: UNSW-NB15_3.csv
Loading UNSW file: UNSW-NB15_4.csv
UNSW loaded: 2,540,047 rows


In [10]:
print('Cleaning UNSW-NB15...')

df_u = unsw_raw.copy()

df_u['attack_cat'] = df_u['attack_cat'].fillna('Normal')
df_u['attack_cat'] = df_u['attack_cat'].astype(str).str.strip()
df_u['attack_cat'] = df_u['attack_cat'].replace({
    '': 'Normal',
    'nan': 'Normal',
    ' ': 'Normal'
})

numeric_cols_unsw = [
    'dsport', 'dur', 'sbytes', 'dbytes', 'Sload', 'Dload',
    'Spkts', 'Dpkts', 'swin', 'dwin', 'smeansz', 'dmeansz',
    'Sjit', 'Djit', 'Sintpkt', 'Dintpkt', 'ct_srv_src',
    'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm'
]

for col in numeric_cols_unsw:
    df_u[col] = pd.to_numeric(df_u[col], errors='coerce')

df_u = df_u.replace([np.inf, -np.inf], np.nan)
df_u = df_u.dropna(subset=numeric_cols_unsw)
df_u = df_u.drop_duplicates()

print(f'UNSW cleaned: {len(df_u):,} rows')
print(df_u['attack_cat'].value_counts())

del unsw_raw
gc.collect()

Cleaning UNSW-NB15...
UNSW cleaned: 2,059,120 rows
attack_cat
Normal            1959477
Exploits            27599
Generic             25378
Fuzzers             21795
Reconnaissance      13357
DoS                  5665
Analysis             2184
Backdoor             1684
Shellcode            1511
Backdoors             299
Worms                 171
Name: count, dtype: int64


0

In [11]:
unsw_label_map = {
    'Normal': 'NORMAL',

    'DoS': 'DoS',

    'Reconnaissance': 'Probe',
    'Fuzzers': 'Probe',
    'Generic': 'Probe',
    'Analysis': 'Probe',

    'Exploits': 'Exploitation',
    'Shellcode': 'Exploitation',
    'Worms': 'Exploitation',

    'Backdoors': 'Backdoor',
    'Backdoor': 'Backdoor',
}

df_u['unified_label'] = df_u['attack_cat'].map(unsw_label_map)

if df_u['unified_label'].isna().sum() > 0:
    print('Unmapped UNSW labels:')
    print(df_u[df_u['unified_label'].isna()]['attack_cat'].value_counts())

df_u = df_u.dropna(subset=['unified_label']).reset_index(drop=True)

print(df_u['unified_label'].value_counts())

unified_label
NORMAL          1959477
Probe             62714
Exploitation      29281
DoS                5665
Backdoor           1983
Name: count, dtype: int64


In [12]:
df_u = stratified_sample(
    df_u,
    label_col='unified_label',
    n_rows=ROWS_PER_DATASET,
    random_state=RANDOM_STATE
)

print(f'UNSW sampled: {len(df_u):,} rows')
print(df_u['unified_label'].value_counts())

UNSW sampled: 99,998 rows
unified_label
NORMAL          95160
Probe            3045
Exploitation     1422
DoS               275
Backdoor           96
Name: count, dtype: int64


/tmp/ipykernel_1406/3033405128.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


In [13]:
print('Extracting UNSW canonical features...')

df_u['dur_us'] = df_u['dur'] * 1_000_000

df_u['flow_bytes_per_sec_derived'] = (df_u['Sload'] + df_u['Dload']) / 8

df_u['flow_pkts_per_sec_derived'] = (
    (df_u['Spkts'] + df_u['Dpkts']) / df_u['dur'].replace(0, np.nan)
)

proto_map = {
    'tcp': 6,
    'udp': 17,
    'icmp': 1
}

df_u['proto_numeric'] = (
    df_u['proto']
    .astype(str)
    .str.lower()
    .map(proto_map)
    .fillna(0)
    .astype(int)
)

df_u['state_clean'] = df_u['state'].astype(str).str.upper()

df_u['syn_flag_count'] = (
    df_u['state_clean'].str.contains('SYN|CON|REQ', na=False)
).astype(int)

df_u['ack_flag_count'] = (
    df_u['state_clean'].str.contains('FIN|CON|ACC', na=False)
).astype(int)

df_u['rst_flag_count'] = (
    df_u['state_clean'].str.contains('RST', na=False)
).astype(int)

df_u['psh_flag_count'] = 0

unsw_features = pd.DataFrame({
    'dest_port'          : df_u['dsport'].fillna(0),
    'protocol'           : df_u['proto_numeric'],
    'flow_duration'      : df_u['dur_us'],

    'fwd_pkts'           : df_u['Spkts'],
    'bwd_pkts'           : df_u['Dpkts'],
    'fwd_bytes'          : df_u['sbytes'],
    'bwd_bytes'          : df_u['dbytes'],

    'fwd_pkt_len_mean'   : df_u['smeansz'],
    'bwd_pkt_len_mean'   : df_u['dmeansz'],

    'fwd_iat_mean'       : df_u['Sintpkt'],
    'bwd_iat_mean'       : df_u['Dintpkt'],
    'fwd_iat_std'        : df_u['Sjit'],
    'bwd_iat_std'        : df_u['Djit'],

    'flow_bytes_per_sec' : df_u['flow_bytes_per_sec_derived'],
    'flow_pkts_per_sec'  : df_u['flow_pkts_per_sec_derived'],

    'syn_flag_count'     : df_u['syn_flag_count'],
    'ack_flag_count'     : df_u['ack_flag_count'],
    'rst_flag_count'     : df_u['rst_flag_count'],
    'psh_flag_count'     : df_u['psh_flag_count'],

    'fwd_win'            : df_u['swin'],
    'bwd_win'            : df_u['dwin'],

    'fwd_load'           : df_u['Sload'] / 8,
    'bwd_load'           : df_u['Dload'] / 8,

    'ct_srv_src'         : df_u['ct_srv_src'],
    'ct_dst_ltm'         : df_u['ct_dst_ltm'],

    'unified_label'      : df_u['unified_label'],
    'dataset_source'     : 1,
})

unsw_features = unsw_features.replace([np.inf, -np.inf], np.nan)
unsw_features = unsw_features.dropna().reset_index(drop=True)

print(unsw_features.shape)
print(unsw_features['unified_label'].value_counts())

del df_u
gc.collect()

Extracting UNSW canonical features...
(99758, 27)
unified_label
NORMAL          94920
Probe            3045
Exploitation     1422
DoS               275
Backdoor           96
Name: count, dtype: int64


37

In [14]:
print('Merging sampled CICIDS2017 + UNSW-NB15...')

merged = pd.concat(
    [cicids_features, unsw_features],
    ignore_index=True
)

del cicids_features
del unsw_features
gc.collect()

print(f'Merged rows: {len(merged):,}')
print(f'Merged columns: {merged.shape[1]}')
print(merged['dataset_source'].value_counts())
print(merged['unified_label'].value_counts())

Merging sampled CICIDS2017 + UNSW-NB15...
Merged rows: 199,754
Merged columns: 27
dataset_source
0    99996
1    99758
Name: count, dtype: int64
unified_label
NORMAL          178030
DoS              13039
Probe             6642
Exploitation      1869
Backdoor           174
Name: count, dtype: int64


In [15]:
label_encoding = {
    'NORMAL': 0,
    'DoS': 1,
    'Probe': 2,
    'Exploitation': 3,
    'Backdoor': 4
}

merged['label'] = merged['unified_label'].map(label_encoding).astype(int)

print('Label encoding:')
for name, number in label_encoding.items():
    print(f'{number}: {name}')

print(merged['label'].value_counts())

Label encoding:
0: NORMAL
1: DoS
2: Probe
3: Exploitation
4: Backdoor
label
0    178030
1     13039
2      6642
3      1869
4       174
Name: count, dtype: int64


In [16]:
merged = merged.replace([np.inf, -np.inf], np.nan)
merged = merged.dropna().reset_index(drop=True)

print('Final quality check')
print(f'Rows: {len(merged):,}')
print(f'Columns: {merged.shape[1]}')
print(f'NaN values: {merged.isna().sum().sum()}')

numeric_cols = merged.select_dtypes(include=[np.number]).columns
inf_rows = np.isinf(merged[numeric_cols]).any(axis=1).sum()

print(f'Infinite rows: {inf_rows}')
print(merged['unified_label'].value_counts())

Final quality check
Rows: 199,754
Columns: 28
NaN values: 0
Infinite rows: 0
unified_label
NORMAL          178030
DoS              13039
Probe             6642
Exploitation      1869
Backdoor           174
Name: count, dtype: int64


In [17]:
final_columns = [
    'dest_port', 'protocol', 'flow_duration',
    'fwd_pkts', 'bwd_pkts', 'fwd_bytes', 'bwd_bytes',
    'fwd_pkt_len_mean', 'bwd_pkt_len_mean',
    'fwd_iat_mean', 'bwd_iat_mean', 'fwd_iat_std', 'bwd_iat_std',
    'flow_bytes_per_sec', 'flow_pkts_per_sec',
    'syn_flag_count', 'ack_flag_count', 'rst_flag_count', 'psh_flag_count',
    'fwd_win', 'bwd_win',
    'fwd_load', 'bwd_load',
    'ct_srv_src', 'ct_dst_ltm',
    'dataset_source',
    'label',
    'unified_label',
]

merged_final = merged[final_columns].copy()

csv_path = os.path.join(OUTPUT_FOLDER, 'umurinzi_merged_800k.csv')
parquet_path = os.path.join(OUTPUT_FOLDER, 'umurinzi_merged_800k.parquet')

merged_final.to_csv(csv_path, index=False)
merged_final.to_parquet(parquet_path, index=False)

print('Saved files:')
print(csv_path)
print(parquet_path)

print(f'Rows: {len(merged_final):,}')
print(f'Columns: {merged_final.shape[1]}')
print(merged_final['unified_label'].value_counts())

Saved files:
/content/drive/MyDrive/umurinzi_data/output/umurinzi_merged_800k.csv
/content/drive/MyDrive/umurinzi_data/output/umurinzi_merged_800k.parquet
Rows: 199,754
Columns: 28
unified_label
NORMAL          178030
DoS              13039
Probe             6642
Exploitation      1869
Backdoor           174
Name: count, dtype: int64
